In [6]:
from sys import displayhook

#import bibliothèque de base
import pandas as pd
import numpy as np

In [7]:
# charger les data
#nba shot locations
df = pd.read_csv('../data/NBA shot locations/NBA_Shot_Locations_1997_2020.csv')
print("Lignes   :", f"{df.shape[0]:,}")  # shape = (nombre de lignes, nombre de colonnes)
print("Colonnes :", df.shape[1])


Lignes   : 4,729,512
Colonnes : 22


In [8]:
display(df.head())


,Game ID,Game Event ID,Player ID,Player Name,Team ID,Team Name,Period,Minutes Remaining,Seconds Remaining,Action Type,...,Shot Zone Area,Shot Zone Range,Shot Distance,X Location,Y Location,Shot Made Flag,Game Date,Home Team,Away Team,Season Type
0,29700427,389,100,Tim Legler,1610612764,Washington Wizards,4,11,22,Jump Shot,...,Right Side(R),8-16 ft.,15,117,109,1,19980102,WAS,IND,Regular Season
1,29700427,406,100,Tim Legler,1610612764,Washington Wizards,4,9,36,Jump Shot,...,Right Side(R),8-16 ft.,14,143,25,0,19980102,WAS,IND,Regular Season
2,29700427,475,100,Tim Legler,1610612764,Washington Wizards,4,3,7,Jump Shot,...,Left Side(L),8-16 ft.,10,-87,55,0,19980102,WAS,IND,Regular Season
3,29700427,487,100,Tim Legler,1610612764,Washington Wizards,4,1,45,Jump Shot,...,Center(C),Less Than 8 ft.,5,-1,53,0,19980102,WAS,IND,Regular Season
4,29700427,497,100,Tim Legler,1610612764,Washington Wizards,4,0,45,Jump Shot,...,Right Side(R),8-16 ft.,14,89,113,0,19980102,WAS,IND,Regular Season


In [9]:
 # les 3 premières lignes ; .T les couche pour lire une colonne par ligne
df.head(3).T

,0,1,2
Game ID,29700427,29700427,29700427
Game Event ID,389,406,475
Player ID,100,100,100
Player Name,Tim Legler,Tim Legler,Tim Legler
Team ID,1610612764,1610612764,1610612764
Team Name,Washington Wizards,Washington Wizards,Washington Wizards
Period,4,4,4
Minutes Remaining,11,9,3
Seconds Remaining,22,36,7
Action Type,Jump Shot,Jump Shot,Jump Shot


In [ ]:
#Game Date est un entier 19980102 (une date déguisée en nombre, à convertir),
# et X Location peut être négatif (−87 = côté gauche, le panier est à l'origine — normal, pas une erreur).

In [10]:
#Le profil de chaque colonne
profil = pd.DataFrame({
    "type":        df.dtypes.astype(str),               # type informatique de chaque colonne
    "NA_%":        (df.isna().mean() * 100).round(2),   # isna() → vrai/faux par case ; mean() → proportion de vrais
    "nb_distinct": df.nunique(),                        # nombre de valeurs différentes
    "exemple":     df.iloc[0].astype(str),              # première valeur, pour se repérer
})
profil

,type,NA_%,nb_distinct,exemple
Game ID,int64,0.0,28817,29700427
Game Event ID,int64,0.0,899,389
Player ID,int64,0.0,2152,100
Player Name,str,0.0,2143,Tim Legler
Team ID,int64,0.0,30,1610612764
Team Name,str,0.0,37,Washington Wizards
Period,int64,0.0,8,4
Minutes Remaining,int64,0.0,13,11
Seconds Remaining,int64,0.0,60,22
Action Type,str,0.0,70,Jump Shot


In [ ]:
#Aucune valeur manquante, nulle part.

#Le nombre de valeurs distinctes classe les colonnes. 2 valeurs → binaire (Shot Made Flag, Shot Type, Season Type).
# Entre 5 et 70 → catégorie (Shot Zone Basic, Action Type…).

# Team ID a 30 valeurs mais Team Name en a 37. si des franchises ont changé de nom nom, l'identifiant est plus fiable que le nom

In [11]:
#verification doublon
print("Lignes strictement identiques :", df.duplicated().sum())
print("Même (Game ID, Game Event ID) :", df.duplicated(subset=["Game ID", "Game Event ID"]).sum())

Lignes strictement identiques : 0
Même (Game ID, Game Event ID) : 0


**Interprétation.** Aucun doublon. Le couple (Game ID, Game Event ID) est une clé unique du tir : on pourra joindre d'autres tables dessus sans multiplier les lignes.

**distribution des variable categorielle**

In [14]:
for col in ["Shot Made Flag", "Shot Type", "Season Type", "Period", "Shot Zone Basic", "Shot Zone Area", "Shot Zone Range"]:
    print(f"\n=== {col} ===")
    effectifs = df[col].value_counts()                                # nombre de tirs par catégorie
    parts     = (df[col].value_counts(normalize=True) * 100).round(1) # part en %
    print(pd.DataFrame({"nb_tirs": effectifs, "part_%": parts}))


=== Shot Made Flag ===
                nb_tirs  part_%
Shot Made Flag                 
0               2592798    54.8
1               2136714    45.2

=== Shot Type ===
                nb_tirs  part_%
Shot Type                      
2PT Field Goal  3606005    76.2
3PT Field Goal  1123507    23.8

=== Season Type ===
                nb_tirs  part_%
Season Type                    
Regular Season  4447109    94.0
Playoffs         282403     6.0

=== Period ===
        nb_tirs  part_%
Period                 
1       1241342    26.2
2       1179133    24.9
3       1157160    24.5
4       1117611    23.6
5         29324     0.6
6          4115     0.1
7           720     0.0
8           107     0.0

=== Shot Zone Basic ===
                       nb_tirs  part_%
Shot Zone Basic                       
Restricted Area        1505921    31.8
Mid-Range              1417625    30.0
Above the Break 3       828349    17.5
In The Paint (Non-RA)   683748    14.5
Left Corner 3           147043     3.

**Interprétation.** Taux de réussite global 45,2 %. Trois quarts des tirs sont à 2 points. Les prolongations (Period 5 à 8) ne font que 0,7 % des tirs. Un tir sur trois part de la zone restreinte, un sur trois de mi-distance. Backcourt (0,2 %) = tirs désespérés depuis l'autre moitié du terrain, cas à part. Shot Zone Area : plus d'un tir sur deux (52,8 %) part de l'axe central du terrain ; les côtés gauche (13,0 %) et droit (12,0 %) sont presque symétriques, de même que les zones intermédiaires (10,9 % / 11,1 %).

**Distribution variables quantitatives.**
 les statistiques descriptives de base : `describe()` donne effectif, moyenne, écart-type, min, quartiles, max. On convertit aussi Game Date en vraie date pour connaître la période couverte.

In [15]:
df["Game Date"] = pd.to_datetime(df["Game Date"].astype(str), format="%Y%m%d")   # 19980102 → 1998-01-02
print("Période couverte :", df["Game Date"].min().date(), "→", df["Game Date"].max().date())

df[["Shot Distance", "X Location", "Y Location", "Minutes Remaining", "Seconds Remaining"]].describe().round(1).T

Période couverte : 1997-10-31 → 2020-03-11


,count,mean,std,min,25%,50%,75%,max
Shot Distance,4729512.0,12.1,9.9,0.0,2.0,13.0,21.0,89.0
X Location,4729512.0,-1.3,110.1,-250.0,-60.0,0.0,56.0,250.0
Y Location,4729512.0,76.3,87.0,-52.0,3.0,38.0,145.0,884.0
Minutes Remaining,4729512.0,5.3,3.5,0.0,2.0,5.0,8.0,12.0
Seconds Remaining,4729512.0,28.7,17.5,0.0,14.0,29.0,44.0,59.0


**Interprétation.** Distance moyenne 12,1 pieds (≈ 4 m), médiane 13 ; le max de 89 pieds correspond à des tirs de l'autre bout du terrain. X est centré sur 0 (−250 à 250 = 50 pieds de large, unités en 1/10 de pied) ; Y va de −52 (derrière le panier) à 884. La période va du 31 oct. 1997 (début de saison 1997-98) au 11 mars 2020 (arrêt COVID). Game Date est maintenant une vraie date, exploitable pour dériver la saison.

**A savoir ** Une saison NBA va d'octobre à juin : un match de janvier 2005 appartient à la saison 2004-05. On dérive donc la saison de la date (année de début), puis on compte les tirs par saison pour vérifier la couverture — c'est ici qu'on repère les trous du dataset.

In [20]:
annee = df["Game Date"].dt.year
mois  = df["Game Date"].dt.month
df["Saison"] = annee.where(mois >= 10, annee - 1)
# oct-déc → année en cours ; jan-juin → année précédente

par_saison = df.groupby("Saison")["Shot Made Flag"].agg(nb_tirs="count", pt_reussite="mean")
par_saison["pt_reussite"] = (par_saison["pt_reussite"] * 100).round(1)
par_saison

,nb_tirs,pt_reussite
Saison,,
1997,200057,45.0
1998,123234,43.6
1999,206790,44.8
2000,203056,44.2
2001,204596,44.4
2002,206114,44.2
2003,202420,43.8
2004,210915,44.8
2005,208085,45.4


**Interprétation.** 23 saisons, environ 200 000 à 234 000 tirs chacune — sauf trois nettement plus basses : **1998-99 (123 k), 2011-12 (174 k), 2019-20 (172 k)**. Ce ne sont pas des erreurs du fichier mais l'histoire de la NBA : deux *lock-outs* (saisons raccourcies à 50 et 66 matchs) et l'arrêt COVID en mars 2020. Conséquence : tout graphique « par saison » en volume sera trompeur pour ces trois années → il faudra raisonner en **part** (%) et non en nombre, ou signaler ces saisons. Deux tendances de fond : le volume de tirs par saison augmente (rythme de jeu plus élevé) et le taux de réussite monte doucement de 44 % à 46 %. Pour le périmètre « XXIe siècle », les saisons 1997, 1998 et 1999 sortent.

In [21]:
#analyse des actions types (70 categories°
actions = df.groupby("Action Type")["Shot Made Flag"].agg(nb_tirs="count", pct_reussite="mean")
actions["pct_reussite"] = (actions["pct_reussite"] * 100).round(1)
actions.sort_values("nb_tirs", ascending=False)

,nb_tirs,pct_reussite
Action Type,,
Jump Shot,2570552,34.5
Layup Shot,609190,47.2
Driving Layup Shot,293150,65.2
Pullup Jump shot,137984,50.1
Hook Shot,91485,47.0
...,...,...
Driving Reverse Dunk Shot,217,87.6
Running Bank Hook Shot,187,83.4
Turnaround Finger Roll Shot,127,83.5


**Interprétation.** Trois constats. (1) **Très déséquilibré** : Jump Shot seul = 54 % des tirs ; 25 catégories ont moins de 5 000 tirs, 10 moins de 500 → à regrouper en 5-6 familles (Jump Shot, Layup, Dunk, Hook, Tip/Putback, autres) pour les graphiques et le modèle. (2) **Le taux de réussite varie énormément selon le geste** : 34,5 % pour un Jump Shot, 65 % pour un Driving Layup, 87 % pour un Dunk, 97,6 % pour un Slam Dunk. C'est une variable très informative (H2 est déjà quasi confirmée à l'œil — on la testera). (3) **Disponibilité a priori** : les libellés décrivent le geste *tenté* (driving, pullup, fadeaway…), pas son issue — sauf un cas douteux, **« No Shot » (278 lignes, 54,7 % de réussite)**, incohérent par nature (un tir enregistré comme « pas de tir »). À exclure ou à signaler. Point de vigilance pour le modèle : les dunks réussissent à 90 %+ parce qu'un dunk *raté* est rare et souvent enregistré autrement — l'annotation du geste n'est pas totalement indépendante du résultat. À noter comme limite.

In [22]:
#analyse inchorence identifiant et nom teams
noms_par_id = df.groupby("Team ID")["Team Name"].unique()     # pour chaque ID, la liste des noms rencontrés
noms_par_id[noms_par_id.apply(len) > 1]                       # ne garder que les ID qui ont plus d'un nom

Team ID
1610612740    [New Orleans Hornets, New Orleans/Oklahoma Cit...
1610612746                  [Los Angeles Clippers, LA Clippers]
1610612751                     [New Jersey Nets, Brooklyn Nets]
1610612760         [Seattle SuperSonics, Oklahoma City Thunder]
1610612763             [Vancouver Grizzlies, Memphis Grizzlies]
1610612766               [Charlotte Hornets, Charlotte Bobcats]
Name: Team Name, dtype: object

**Interprétation.** Six franchises ont changé de nom ou de ville entre 1997 et 2020 (déménagements Seattle → Oklahoma City, Vancouver → Memphis, New Jersey → Brooklyn ; renommages Hornets/Bobcats/Pelicans ; simple variante « LA Clippers »). L'écart 37/30 est entièrement expliqué. Conséquence pratique : pour toute analyse par équipe, utiliser **Team ID** (stable) et non Team Name ; c'est aussi la clé à utiliser pour joindre les autres tables Kaggle, qui emploient les mêmes ID.